# EDA: Feature Opportunity Analysis

**Goal:** Identify new feature engineering opportunities to push F1 beyond 62.5% toward 75%

## Questions to Answer:
1. How separable are the classes in current feature space? (Is 75% F1 realistic?)
2. Which features have the strongest mutual information with the target?
3. Is there distribution shift between train (2015-2017) and test (2018-2020)?
4. What do false positives and false negatives look like?
5. Are there interaction effects not yet captured?
6. Is there text signal beyond TF-IDF?

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')

%matplotlib inline

figures_dir = Path('../reports/figures')
figures_dir.mkdir(parents=True, exist_ok=True)
print('Setup complete')

## 1. Load Data

In [ ]:
# Load cleaned dataset
df = pd.read_pickle('../data/processed/cleaned_data.pkl')

# Load features and targets
X_train = pd.read_pickle('../data/features/X_train_temporal.pkl')
X_test  = pd.read_pickle('../data/features/X_test_temporal.pkl')
y_train = pd.read_pickle('../data/features/y_train_cls_temporal.pkl')
y_test  = pd.read_pickle('../data/features/y_test_cls_temporal.pkl')
metadata_train = pd.read_pickle('../data/features/metadata_train.pkl')
metadata_test  = pd.read_pickle('../data/features/metadata_test.pkl')

# Load venue and author features separately for targeted analysis
venue_features  = pd.read_pickle('../data/features/venue_features.pkl')
author_features = pd.read_pickle('../data/features/author_features.pkl')

# Align raw data to feature indices
all_idx = list(X_train.index) + list(X_test.index)
df_aligned = df.loc[all_idx]
y_all = pd.concat([y_train, y_test])

print(f'Cleaned data: {df.shape}')
print(f'Train features: {X_train.shape}, Test features: {X_test.shape}')
print(f'Columns in cleaned data: {list(df.columns)}')

## 2. Class Separability: Is 75% F1 Achievable?

We use PCA + a 2D visualization to see if high-impact and low-impact papers form distinct clusters.

In [ ]:
# Use non-TF-IDF features only for visualization (venue + author)
structured_cols = [c for c in X_train.columns if not c.startswith('tfidf_')]
X_struct_train = X_train[structured_cols]
X_struct_test  = X_test[structured_cols]

print(f'Structured features (non-TF-IDF): {len(structured_cols)}')
print(structured_cols)

In [ ]:
# PCA on structured features
scaler = StandardScaler()
X_struct_all = pd.concat([X_struct_train, X_struct_test])
X_scaled = scaler.fit_transform(X_struct_all.fillna(0))

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#2196F3' if y == 0 else '#F44336' for y in y_all.values]
ax.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.4, s=20)
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2196F3', label='Low-impact'),
    Patch(facecolor='#F44336', label='High-impact')
]
ax.legend(handles=legend_elements, loc='upper right')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('PCA of Structured Features: High vs Low Impact Papers')
plt.tight_layout()
plt.savefig(figures_dir / 'pca_class_separability.png', dpi=200, bbox_inches='tight')
plt.show()

print(f'Total variance explained by 2 PCs: {sum(pca.explained_variance_ratio_)*100:.1f}%')

## 3. Mutual Information: Which Features Actually Matter?

Rank ALL structured features by mutual information with the target.

In [ ]:
X_mi = X_struct_all.fillna(0)
mi_scores = mutual_info_classif(X_mi, y_all, random_state=42)

mi_df = pd.DataFrame({
    'feature': X_struct_all.columns,
    'mutual_info': mi_scores
}).sort_values('mutual_info', ascending=False)

print('Mutual Information Ranking (all structured features):')
print(mi_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(mi_df['feature'][::-1], mi_df['mutual_info'][::-1], color='steelblue')
ax.set_xlabel('Mutual Information with Target')
ax.set_title('Feature Predictive Power (Mutual Information)')
plt.tight_layout()
plt.savefig(figures_dir / 'mutual_information_ranking.png', dpi=200, bbox_inches='tight')
plt.show()

## 4. Distribution Shift: Train vs Test

If feature distributions differ between 2015-2017 (train) and 2018-2020 (test),
the model cannot generalize — this is a root cause of poor F1.

In [ ]:
from scipy.stats import ks_2samp

shift_results = []
for col in structured_cols:
    train_vals = X_struct_train[col].dropna()
    test_vals  = X_struct_test[col].dropna()
    stat, pval = ks_2samp(train_vals, test_vals)
    shift_results.append({
        'feature': col,
        'ks_statistic': stat,
        'p_value': pval,
        'significant_shift': pval < 0.05
    })

shift_df = pd.DataFrame(shift_results).sort_values('ks_statistic', ascending=False)

print('Distribution Shift (KS Test, Train 2015-17 vs Test 2018-20):')
print(shift_df.to_string(index=False))

n_shifted = shift_df['significant_shift'].sum()
print(f'\nFeatures with significant distribution shift: {n_shifted} / {len(shift_df)}')
print('HIGH SHIFT = model may be learning temporal artifacts, not true signal')

In [ ]:
# Plot distributions for top-shifted features
top_shifted = shift_df[shift_df['significant_shift']]['feature'].tolist()[:6]

if top_shifted:
    n_cols = min(3, len(top_shifted))
    n_rows = (len(top_shifted) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    for i, feat in enumerate(top_shifted):
        ax = axes[i]
        train_vals = X_struct_train[feat].dropna()
        test_vals  = X_struct_test[feat].dropna()
        ax.hist(train_vals, bins=30, alpha=0.5, label='Train (2015-17)', color='steelblue', density=True)
        ax.hist(test_vals,  bins=30, alpha=0.5, label='Test (2018-20)',  color='coral',    density=True)
        ks_stat = shift_df[shift_df['feature'] == feat]['ks_statistic'].values[0]
        ax.set_title(f'{feat}\n(KS={ks_stat:.3f})')
        ax.legend(fontsize=8)
    
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Feature Distribution: Train vs Test', y=1.01)
    plt.tight_layout()
    plt.savefig(figures_dir / 'distribution_shift.png', dpi=200, bbox_inches='tight')
    plt.show()
else:
    print('No significant distribution shift detected — good!')

## 5. Feature Distributions by Class

Box plots of each structured feature split by high-impact vs low-impact.

In [ ]:
# Combine train + test with labels for full picture
X_all_struct = pd.concat([X_struct_train, X_struct_test])
y_all_aligned = pd.concat([y_train, y_test])

analysis_df = X_all_struct.copy()
analysis_df['high_impact'] = y_all_aligned.values
analysis_df['split'] = ['train'] * len(X_struct_train) + ['test'] * len(X_struct_test)

# Summary stats by class
print('Feature Statistics by Class (High vs Low Impact):')
print('=' * 80)

class_stats = []
for col in structured_cols:
    low  = analysis_df[analysis_df['high_impact'] == 0][col]
    high = analysis_df[analysis_df['high_impact'] == 1][col]
    class_stats.append({
        'feature': col,
        'low_median':  low.median(),
        'high_median': high.median(),
        'ratio': high.median() / (low.median() + 1e-9),
        'low_mean':  low.mean(),
        'high_mean': high.mean(),
    })

class_stats_df = pd.DataFrame(class_stats).sort_values('ratio', ascending=False)
print(class_stats_df.to_string(index=False))

In [ ]:
# Box plots for top discriminating features
top_features_by_ratio = class_stats_df.head(min(12, len(class_stats_df)))['feature'].tolist()

n_cols = 3
n_rows = (len(top_features_by_ratio) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for i, feat in enumerate(top_features_by_ratio):
    ax = axes[i]
    data_low  = analysis_df[analysis_df['high_impact'] == 0][feat].dropna()
    data_high = analysis_df[analysis_df['high_impact'] == 1][feat].dropna()
    ax.boxplot([data_low, data_high], labels=['Low', 'High'], patch_artist=True,
               boxprops=dict(facecolor='lightblue'))
    ax.set_title(feat, fontsize=9)
    ax.set_ylabel('Value')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions: Low-Impact vs High-Impact Papers', y=1.01)
plt.tight_layout()
plt.savefig(figures_dir / 'feature_distributions_by_class.png', dpi=200, bbox_inches='tight')
plt.show()

## 6. Interaction Effects

Test candidate interaction features to see if they separate classes better than individual features.

In [ ]:
# Candidate interactions based on domain knowledge
# h_index × venue prestige, reference count × page count, etc.

def safe_col(df, name):
    return df[name] if name in df.columns else pd.Series(np.zeros(len(df)), index=df.index)

interactions = {}

# h_index × venue prestige
if 'h_index_max' in X_all_struct.columns and 'snip' in X_all_struct.columns:
    interactions['h_index_x_snip'] = X_all_struct['h_index_max'] * X_all_struct['snip']
if 'h_index_max' in X_all_struct.columns and 'citescore' in X_all_struct.columns:
    interactions['h_index_x_citescore'] = X_all_struct['h_index_max'] * X_all_struct['citescore']

# author count × international collaboration
if 'num_authors' in X_all_struct.columns and 'is_international_collab' in X_all_struct.columns:
    interactions['authors_x_international'] = X_all_struct['num_authors'] * X_all_struct['is_international_collab']

# top journal × high h-index
if 'is_top_journal' in X_all_struct.columns and 'h_index_max' in X_all_struct.columns:
    interactions['top_journal_x_high_hindex'] = X_all_struct['is_top_journal'] * X_all_struct['h_index_max']

# snip percentile × citescore percentile
if 'snip_percentile' in X_all_struct.columns and 'citescore_percentile' in X_all_struct.columns:
    interactions['snip_pct_x_citescore_pct'] = X_all_struct['snip_percentile'] * X_all_struct['citescore_percentile']

# num_institutions × num_countries
if 'num_institutions' in X_all_struct.columns and 'num_countries' in X_all_struct.columns:
    interactions['institutions_x_countries'] = X_all_struct['num_institutions'] * X_all_struct['num_countries']

print(f'Created {len(interactions)} interaction features:')
for name in interactions:
    print(f'  - {name}')

In [ ]:
if interactions:
    interaction_df_all = pd.DataFrame(interactions)
    interaction_df_all['high_impact'] = y_all_aligned.values

    int_mi = mutual_info_classif(
        interaction_df_all.drop('high_impact', axis=1).fillna(0),
        interaction_df_all['high_impact'],
        random_state=42
    )

    int_mi_df = pd.DataFrame({
        'interaction': list(interactions.keys()),
        'mutual_info': int_mi
    }).sort_values('mutual_info', ascending=False)

    print('Mutual Information of Interaction Features vs Individual Features:')
    print('\nInteractions:')
    print(int_mi_df.to_string(index=False))
    print('\nTop individual features (for comparison):')
    print(mi_df.head(len(int_mi_df)).to_string(index=False))

## 7. Explore Raw Columns Not Yet Used

Some columns in cleaned_data.pkl may not be fully exploited.

In [ ]:
# Which columns exist in cleaned data but aren't in current features?
used_raw_cols = [
    'Abstract', 'Title', 'Year', 'Citations', 'Authors',
    'Number of Authors', 'Number of Institutions', 'Number of Countries/Regions',
    'Authors H-index', 'Scopus Source title',
    'SNIP (publication year)', 'SNIP percentile (publication year) *',
    'CiteScore (publication year)', 'CiteScore percentile (publication year) *',
    'SJR (publication year)', 'SJR percentile (publication year) *',
    'Open Access', 'Publication type', 'Topic Prominence Percentile',
    'Source type', 'Language',
    'Topic Cluster Prominence Percentile', 'Publication link to Topic strength',
    'Topic Cluster name', 'Pages', 'Reference',
    'All Science Journal Classification (ASJC) field name',
    'Publisher', 'Sustainable Development Goals (2025)', 'EID'
]

all_cols = df.columns.tolist()
unused_cols = [c for c in all_cols if c not in used_raw_cols]

print(f'All columns in cleaned data: {len(all_cols)}')
print(f'Already used/analyzed: {len(used_raw_cols)}')
print(f'Potentially unused columns: {len(unused_cols)}')
print()
for col in unused_cols:
    n_missing = df[col].isna().sum()
    pct_missing = n_missing / len(df) * 100
    sample_vals = df[col].dropna().unique()[:5]
    print(f'  [{pct_missing:5.1f}% missing] {col}: {sample_vals}')

## 8. Text Signal Analysis: High vs Low Impact Abstracts

Beyond TF-IDF — do high-impact abstracts have structural/linguistic patterns?

In [ ]:
df_with_label = df.loc[all_idx].copy()
df_with_label['high_impact'] = y_all.values

# Text-based features from Abstract
df_with_label['abstract_len_chars'] = df_with_label['Abstract'].str.len()
df_with_label['abstract_word_count'] = df_with_label['Abstract'].str.split().str.len()
df_with_label['abstract_sentence_count'] = df_with_label['Abstract'].str.count(r'[.!?]+')
df_with_label['abstract_avg_word_len'] = (
    df_with_label['Abstract'].str.replace(r'[^a-zA-Z ]', '', regex=True)
    .apply(lambda t: np.mean([len(w) for w in str(t).split()]) if str(t).split() else 0)
)

# Method/results keywords (proxy for methodology richness)
method_keywords = ['method', 'approach', 'algorithm', 'model', 'technique', 'framework', 'propose', 'novel']
result_keywords = ['result', 'outperform', 'improve', 'achieve', 'significant', 'state-of-the-art', 'demonstrate']
data_keywords   = ['dataset', 'experiment', 'evaluation', 'benchmark', 'comparison', 'validation']

for kw_list, kw_name in [(method_keywords, 'method'), (result_keywords, 'result'), (data_keywords, 'data')]:
    pattern = '|'.join(kw_list)
    df_with_label[f'abstract_{kw_name}_kw_count'] = (
        df_with_label['Abstract'].str.lower().str.count(pattern)
    )

# Title features
df_with_label['title_word_count'] = df_with_label['Title'].str.split().str.len()
df_with_label['title_has_colon']  = df_with_label['Title'].str.contains(':').astype(int)
df_with_label['title_has_question'] = df_with_label['Title'].str.contains(r'\?').astype(int)

text_derived_cols = [
    'abstract_len_chars', 'abstract_word_count', 'abstract_sentence_count',
    'abstract_avg_word_len', 'abstract_method_kw_count',
    'abstract_result_kw_count', 'abstract_data_kw_count',
    'title_word_count', 'title_has_colon', 'title_has_question'
]

print('Text-Derived Feature Statistics by Class:')
print('=' * 70)
for col in text_derived_cols:
    low_med  = df_with_label[df_with_label['high_impact'] == 0][col].median()
    high_med = df_with_label[df_with_label['high_impact'] == 1][col].median()
    ratio = high_med / (low_med + 1e-9)
    print(f'  {col:<35s}  low={low_med:.2f}  high={high_med:.2f}  ratio={ratio:.2f}')

In [ ]:
# MI for text-derived features
text_feat_df = df_with_label[text_derived_cols].fillna(0)
text_mi = mutual_info_classif(text_feat_df, df_with_label['high_impact'], random_state=42)

text_mi_df = pd.DataFrame({
    'feature': text_derived_cols,
    'mutual_info': text_mi
}).sort_values('mutual_info', ascending=False)

print('Mutual Information: Text-Derived Features')
print(text_mi_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(text_mi_df['feature'][::-1], text_mi_df['mutual_info'][::-1], color='teal')
ax.set_xlabel('Mutual Information with Target')
ax.set_title('Text-Derived Feature Predictive Power')
plt.tight_layout()
plt.savefig(figures_dir / 'text_feature_mutual_info.png', dpi=200, bbox_inches='tight')
plt.show()

## 9. Error Analysis: What Do Misclassified Papers Look Like?

Train a simple model, find false positives/negatives, and profile them.

In [ ]:
import pickle

# Load saved LightGBM classifier
try:
    with open('../models/classification/lightgbm.pkl', 'rb') as f:
        clf = pickle.load(f)
    y_pred_proba = clf.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.54).astype(int)
    print('Loaded saved LightGBM model')
except Exception as e:
    print(f'Could not load saved model ({e}), training fresh LR...')
    lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42, n_jobs=-1)
    lr.fit(X_train, y_train)
    y_pred_proba = lr.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.54).astype(int)

print(f'F1 on test set: {f1_score(y_test, y_pred)*100:.2f}%')

In [ ]:
# Build error analysis dataframe
error_df = metadata_test.copy()
error_df['actual']      = y_test.values
error_df['predicted']   = y_pred
error_df['probability'] = y_pred_proba
error_df['error_type']  = 'Correct'
error_df.loc[(error_df['actual'] == 1) & (error_df['predicted'] == 0), 'error_type'] = 'False Negative'
error_df.loc[(error_df['actual'] == 0) & (error_df['predicted'] == 1), 'error_type'] = 'False Positive'

# Add raw data features for profiling
raw_cols_for_analysis = ['Citations']
optional_cols = [
    'Publication type', 'Source type', 'Language', 'Open Access',
    'Reference', 'Pages', 'Topic Prominence Percentile',
    'All Science Journal Classification (ASJC) field name'
]
for col in optional_cols:
    if col in df.columns:
        raw_cols_for_analysis.append(col)

test_raw = df.loc[X_test.index, raw_cols_for_analysis]
error_df = error_df.join(test_raw)

print('Error Type Distribution:')
print(error_df['error_type'].value_counts())
print()
print('Error breakdown by Year:')
print(pd.crosstab(error_df['Year'], error_df['error_type']))

In [ ]:
# Profile false positives vs false negatives
fp = error_df[error_df['error_type'] == 'False Positive']
fn = error_df[error_df['error_type'] == 'False Negative']
correct = error_df[error_df['error_type'] == 'Correct']

print('=== FALSE POSITIVES (predicted high, actually low) ===')
print(f'Count: {len(fp)}')
print(f'Mean citations: {fp["Citations"].mean():.1f} (threshold is global 75th pct)')
print(f'Mean probability: {fp["probability"].mean():.3f}')

for col in optional_cols:
    if col in fp.columns and fp[col].notna().any():
        try:
            if fp[col].dtype in ['object', 'category']:
                print(f'\n{col} distribution in FP:')
                print(fp[col].value_counts().head(5).to_string())
            else:
                print(f'\n{col}: FP mean={fp[col].mean():.2f}, FN mean={fn[col].mean():.2f}, Correct mean={correct[col].mean():.2f}')
        except:
            pass

print()
print('=== FALSE NEGATIVES (predicted low, actually high) ===')
print(f'Count: {len(fn)}')
print(f'Mean citations: {fn["Citations"].mean():.1f}')
print(f'Mean probability: {fn["probability"].mean():.3f}')

In [ ]:
# Visualize prediction probability distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Probability distribution by error type
for err_type, color in [('Correct', 'green'), ('False Positive', 'red'), ('False Negative', 'orange')]:
    subset = error_df[error_df['error_type'] == err_type]['probability']
    axes[0].hist(subset, bins=30, alpha=0.5, label=err_type, color=color, density=True)

axes[0].axvline(0.54, color='black', linestyle='--', label='Threshold (0.54)')
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('Density')
axes[0].set_title('Prediction Confidence by Error Type')
axes[0].legend()

# Error rate by year
year_errors = error_df.groupby('Year').apply(
    lambda g: pd.Series({
        'error_rate': (g['error_type'] != 'Correct').mean(),
        'fn_rate': (g['error_type'] == 'False Negative').mean(),
        'fp_rate': (g['error_type'] == 'False Positive').mean()
    })
).reset_index()

years = year_errors['Year'].values
x = np.arange(len(years))
width = 0.3
axes[1].bar(x - width, year_errors['fn_rate'], width, label='False Neg Rate', color='orange')
axes[1].bar(x,         year_errors['fp_rate'], width, label='False Pos Rate', color='red')
axes[1].bar(x + width, year_errors['error_rate'], width, label='Total Error Rate', color='gray', alpha=0.6)
axes[1].set_xticks(x)
axes[1].set_xticklabels(years)
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Rate')
axes[1].set_title('Error Rates by Year (Test Set)')
axes[1].legend()

plt.tight_layout()
plt.savefig(figures_dir / 'error_analysis.png', dpi=200, bbox_inches='tight')
plt.show()

## 10. Candidate Interaction Feature Test

Add the best interaction features to the existing feature set and measure F1 impact.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

def build_interaction_features(X_struct, df_raw, idx):
    """Build interaction + text-derived features for a given index."""
    feats = pd.DataFrame(index=idx)
    
    # Interaction: h_index × venue prestige
    for v in ['snip', 'citescore', 'sjr']:
        if 'h_index_max' in X_struct.columns and v in X_struct.columns:
            feats[f'h_index_x_{v}'] = X_struct.loc[idx, 'h_index_max'] * X_struct.loc[idx, v]
    
    # Interaction: international collab × h_index
    if 'is_international_collab' in X_struct.columns and 'h_index_max' in X_struct.columns:
        feats['intl_x_h_index'] = X_struct.loc[idx, 'is_international_collab'] * X_struct.loc[idx, 'h_index_max']
    
    # Interaction: snip_pct × citescore_pct
    if 'snip_percentile' in X_struct.columns and 'citescore_percentile' in X_struct.columns:
        feats['snip_pct_x_cs_pct'] = X_struct.loc[idx, 'snip_percentile'] * X_struct.loc[idx, 'citescore_percentile']

    # Text features from abstract
    sub = df_raw.loc[idx, 'Abstract'].fillna('')
    feats['abstract_word_count']  = sub.str.split().str.len()
    feats['abstract_sent_count']  = sub.str.count(r'[.!?]+')
    method_pat = 'method|approach|algorithm|model|technique|framework|propose|novel'
    result_pat = 'result|outperform|improv|achiev|significant|state-of-the-art|demonstrat'
    feats['abstract_method_kw'] = sub.str.lower().str.count(method_pat)
    feats['abstract_result_kw'] = sub.str.lower().str.count(result_pat)

    # Title features
    if 'Title' in df_raw.columns:
        title_sub = df_raw.loc[idx, 'Title'].fillna('')
        feats['title_word_count']  = title_sub.str.split().str.len()
        feats['title_has_colon']   = title_sub.str.contains(':').astype(int)
    
    return feats.fillna(0)

# Build for train and test
X_struct_all_full = pd.concat([X_struct_train, X_struct_test])

extra_train = build_interaction_features(X_struct_all_full, df, X_train.index)
extra_test  = build_interaction_features(X_struct_all_full, df, X_test.index)

X_train_aug = pd.concat([X_train, extra_train], axis=1)
X_test_aug  = pd.concat([X_test, extra_test], axis=1)

print(f'Original features:  {X_train.shape[1]}')
print(f'New interaction features: {extra_train.shape[1]}')
print(f'Augmented features: {X_train_aug.shape[1]}')
print(f'\nNew features: {list(extra_train.columns)}')

In [ ]:
# Baseline
lr_base = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42, n_jobs=-1)
lr_base.fit(X_train, y_train)
y_base_proba = lr_base.predict_proba(X_test)[:, 1]
y_base_pred  = (y_base_proba >= 0.54).astype(int)
f1_base = f1_score(y_test, y_base_pred)

# Augmented
lr_aug = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42, n_jobs=-1)
lr_aug.fit(X_train_aug, y_train)
y_aug_proba = lr_aug.predict_proba(X_test_aug)[:, 1]
y_aug_pred  = (y_aug_proba >= 0.54).astype(int)
f1_aug = f1_score(y_test, y_aug_pred)

print('=' * 60)
print('INTERACTION + TEXT FEATURES IMPACT')
print('=' * 60)
print(f'Baseline F1:    {f1_base*100:.2f}%')
print(f'Augmented F1:   {f1_aug*100:.2f}%')
print(f'Change:         {(f1_aug - f1_base)*100:+.2f} points')

In [ ]:
# Optimize threshold on augmented model
best_thresh, best_f1 = 0.5, 0.0
for t in np.arange(0.40, 0.70, 0.01):
    preds = (y_aug_proba >= t).astype(int)
    f = f1_score(y_test, preds)
    if f > best_f1:
        best_f1 = f
        best_thresh = t

print(f'Best threshold for augmented model: {best_thresh:.2f}')
print(f'Best F1 at optimal threshold:       {best_f1*100:.2f}%')
print(f'Improvement over baseline:          {(best_f1 - f1_base)*100:+.2f} points')

## 11. Summary & Recommendations

In [ ]:
print('=' * 70)
print('EDA SUMMARY: FEATURE OPPORTUNITIES')
print('=' * 70)

print(f'''
1. CLASS SEPARABILITY
   Check PCA plot above. If classes strongly overlap → 75% F1 may be 
   fundamentally difficult with tabular features alone.
   Recommendation: If overlap is high, consider SciBERT embeddings.

2. TOP MUTUAL INFORMATION FEATURES
   The features with highest MI should be prioritized. If venue metrics
   dominate, adding more venue signals (field-level percentile normalization)
   is the highest-leverage engineering effort.

3. DISTRIBUTION SHIFT
   Features with high KS statistic shift between train/test mean the model
   learned temporal patterns. Fix: add year as a feature explicitly, or
   use year-relative normalizations.

4. TEXT SIGNAL
   If abstract_method_kw / abstract_result_kw have high MI, methodologically
   rich abstracts do get cited more — and SciBERT would capture this far
   better than TF-IDF.

5. INTERACTION EFFECTS
   h_index × venue and international × h_index interactions test whether
   the combination of a strong author + strong venue is super-additive.

NEXT STEPS TO REACH 75% F1:
   A. If text MI is high → replace TF-IDF with SciBERT (fastest path)
   B. If distribution shift is severe → retrain with year-aware features
   C. If interactions improve MI → add them to the feature set
   D. If class overlap is too high → accept that the ceiling is ~65% F1
      and focus on framing (e.g. use year-normalized target)
''')
print('=' * 70)